# Multivariate time series and stationary processes: lab notebook

**Statistical Modelling and Forecasting, Multivariate and advanced methods**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import ccf, acf

rng = np.random.default_rng(3)
n = 300

### 1. Two series where x leads y by 4 periods

In [ ]:
# 1. Two series where x leads y by 4 periods
x = np.zeros(n)
for t in range(1, n):
    x[t] = 0.7 * x[t - 1] + rng.standard_normal()
y = np.zeros(n)
y[4:] = 0.9 * x[:-4]
y += rng.standard_normal(n)
df = pd.DataFrame({"x": x, "y": y})
print("mean vector:\n", df.mean().round(3).to_string())
print("covariance matrix:\n", df.cov().round(3))
print("correlation matrix:\n", df.corr().round(3))

### 2. Cross-correlation: corr(x_{t+h}, y_t). statsmodels ccf(a, b)[k] = corr(a_{t+k}, b_t)

In [ ]:
# 2. Cross-correlation: corr(x_{t+h}, y_t). statsmodels ccf(a, b)[k] = corr(a_{t+k}, b_t)
def cc(a, b, K=10):
    fwd = ccf(a, b, adjusted=False)[:K + 1]          # h = 0, 1, ..., K
    bwd = ccf(b, a, adjusted=False)[1:K + 1]         # h = -1, ..., -K
    return np.arange(-K, K + 1), np.concatenate([bwd[::-1], fwd])
h, r = cc(x, y)
band = 1.96 / np.sqrt(n)
print("\nlargest |CCF| at h =", h[np.argmax(np.abs(r))], "value", round(r.max() if abs(r.max()) > abs(r.min()) else r.min(), 3), "| band:", round(band, 3))
print("negative h means x leads y: x_{t-4} is related to y_t")

### 3. Independent random walks: spurious cross-correlation

In [ ]:
# 3. Independent random walks: spurious cross-correlation
a = np.cumsum(rng.standard_normal(n))
b = np.cumsum(rng.standard_normal(n))
h2, r2 = cc(a, b)
print("\nindependent random walks: share of CCF values outside the band:", round(np.mean(np.abs(r2) > band), 2))

### 4. Prewhitening: fit an AR(1) to x, apply the same filter to both

In [ ]:
# 4. Prewhitening: fit an AR(1) to x, apply the same filter to both
phi = sm.OLS(x[1:], x[:-1]).fit().params[0]
xw = x[1:] - phi * x[:-1]
yw = y[1:] - phi * y[:-1]
h3, r3 = cc(xw, yw)
print("\nAR(1) coefficient of x:", round(phi, 3))
print("prewhitened CCF peak at h =", h3[np.argmax(np.abs(r3))], "| values at h = -4, -3, 0:", np.round([r3[h3 == -4][0], r3[h3 == -3][0], r3[h3 == 0][0]], 3))

### 5. Independent AR(1) series: false signals vanish after prewhitening

In [ ]:
# 5. Independent AR(1) series: false signals vanish after prewhitening
p = np.zeros(n); q = np.zeros(n)
for t in range(1, n):
    p[t] = 0.9 * p[t - 1] + rng.standard_normal()
    q[t] = 0.9 * q[t - 1] + rng.standard_normal()
_, rr = cc(p, q)
_, rw = cc(p[1:] - 0.9 * p[:-1], q[1:] - 0.9 * q[:-1])
print("\nindependent AR(1)(0.9): outside the band raw:", round(np.mean(np.abs(rr) > band), 2), "| after prewhitening:", round(np.mean(np.abs(rw) > band), 2))

## Expected output

Running all cells above should print:

```
mean vector:
 x    0.182
y    0.232
covariance matrix:
        x      y
x  1.837  0.513
y  0.513  2.375
correlation matrix:
        x      y
x  1.000  0.246
y  0.246  1.000

largest |CCF| at h = -4 value 0.77 | band: 0.113
negative h means x leads y: x_{t-4} is related to y_t

independent random walks: share of CCF values outside the band: 1.0

AR(1) coefficient of x: 0.672
prewhitened CCF peak at h = -4 | values at h = -4, -3, 0: [0.581 0.046 0.081]

independent AR(1)(0.9): outside the band raw: 0.76 | after prewhitening: 0.1
```

## Practice exercises

Try these in the cells below.

1. Set the lag to 4 with strength 0.8 and read the position of the CCF peak. Which variable leads?
2. Choose "Two independent random walks" and see how many CCF bars exceed the band. Then difference (tick Prewhiten).
3. Make x strongly autocorrelated (φ = 0.9) with no true link, and compare the raw and prewhitened CCF.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
